# start

In [1]:
import os
import pandas as pd
from astropy.io import fits
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
def get_fits_headers(full_file_name, verbose = False):

    try:
        list_hdu = []
        list_tag = []
        list_value = []
        list_description = []
        if verbose:
            print(f"Reading: {full_file_name}")

        with fits.open(full_file_name) as hdul:
            for i, hdu in enumerate(hdul):
                x = repr(hdu.header).split('\n')
                for j in x:
                    list_hdu.append(i)
                    j = j.replace('=',',')
                    j = j.replace('/',',')
                    y = j.split(',')
                    y = [item.strip() for item in y if item.strip()]
                    if len(y) == 2:
                        y.append('')

                    list_tag.append(y[0])
                    list_value.append(y[1])
                    if y[2] == '[km':
                        if y[0] == 'WINDSPEE':
                            y[2] = 'km wind speed'
                    list_description.append(y[2])
        
        return(list_hdu, list_tag, list_value, list_description)
    
    except Exception as e:
        print(f"Error: {e}")
        return None



In [3]:
def get_normalised(list_of_values, verbose = False) :
    try:
        if not list_of_values:
            print("List of values is empty.")
            return [], None, None
        
        if verbose:
            print(f"Original values: {list_of_values}")
            
        min_value = min(list_of_values)
        max_value = max(list_of_values)
        if verbose:
            print(f"Min: {min_value}, Max: {max_value}")
        
        normal_values = [(value - min_value) / (max_value - min_value) for value in list_of_values]
        if verbose:
            print(f"Normalised values: {normal_values}")    
            
        return normal_values, min_value, max_value
        # return [(value - min_value) / (max_value - min_value) for value in list_of_values]
    except Exception as e:
        print(f"Error normalizing values: {e}")
        return [], None, None

# get BANZAI header information as CSV

## note the interpretation of 
[arcsec] Frame FWHM in arcsec header - L1FWHM
“How wide a typical star appears on this image, in terms of angular size on the sky.”
Affected by optics, seeing, focus, tracking

In [46]:
current_dir = os.getcwd()  # Get the current working directory
# star_info = ['WASP-123b','2024-07-01', "lco_data-20250414-660",333,[239]]
star_info = ['HATS-38b','2025-03-26',"lco_data-20250407-211",125,[]]

In [51]:
# file_path = os.path.join(current_dir, "HATS-38b", "lco_data-20250407-211")
star = star_info[0]
obs_date = star_info[1]
folder = star_info[2]
first_frame = star_info[3]
exclude_files = star_info[4]

file_path = os.path.join(current_dir, star, folder)
fits_files = [f for f in os.listdir(file_path) if f.endswith(".fits.fz")]

try:
    for fileno, file_name in enumerate(fits_files):
        if fileno in exclude_files:
            continue
        # Create a DataFrame from the lists
        list_hdu, list_tag, list_value, list_description = get_fits_headers(os.path.join(file_path, file_name))

        if list_hdu is None:
            print(f"Failed to read {file_name}. Skipping.")
            continue

        if fileno == 0:
            df = pd.DataFrame({
                'HDU': list_hdu,
                'Description': list_description,
                'Tag': list_tag,
                f'Frame-{fileno + first_frame}': list_value
            })
        else:
            df = df.copy()    
            df[f'Frame-{fileno + first_frame}'] = list_value

    #df.to_csv(os.path.join(file_path, f"headers-{star}.csv"), index=False)
            
except Exception as e:
        print(f"Error reading {fileno} FITS file: {e}")

# run charts

In [58]:
xtext = '[UTC] Start date and time of the observati'

# get the row with the start and end time
df[df['Description'] == xtext].reset_index(drop=True).iloc[0].tolist()

# xtext = df_xlabs.iloc[0][value_columns].values.astype(str).tolist()
# xtext = [x[12:20] for x in xtext]
# xtext = [f'{value_frames[i]} : {x}' for i, x in enumerate(xtext)]


[np.int64(1),
 '[UTC] Start date and time of the observati',
 'DATE-OBS',
 "'2025-03-26T19:30:13.092'",
 "'2025-03-26T19:31:35.907'",
 "'2025-03-26T19:32:58.763'",
 "'2025-03-26T19:34:21.532'",
 "'2025-03-26T19:35:44.361'",
 "'2025-03-26T19:37:07.214'",
 "'2025-03-26T19:38:30.050'",
 "'2025-03-26T19:39:52.980'",
 "'2025-03-26T19:41:15.820'",
 "'2025-03-26T19:42:38.665'",
 "'2025-03-26T19:44:01.177'",
 "'2025-03-26T19:45:24.022'",
 "'2025-03-26T19:46:46.880'",
 "'2025-03-26T19:48:09.717'",
 "'2025-03-26T19:49:32.565'",
 "'2025-03-26T19:50:55.396'",
 "'2025-03-26T19:52:18.232'",
 "'2025-03-26T19:53:41.057'",
 "'2025-03-26T19:55:03.879'",
 "'2025-03-26T19:56:26.596'",
 "'2025-03-26T19:57:49.435'",
 "'2025-03-26T19:59:12.247'",
 "'2025-03-26T20:00:35.079'",
 "'2025-03-26T20:01:57.919'",
 "'2025-03-26T20:03:20.754'",
 "'2025-03-26T20:04:43.592'",
 "'2025-03-26T20:06:06.438'",
 "'2025-03-26T20:07:29.271'",
 "'2025-03-26T20:08:52.121'",
 "'2025-03-26T20:10:14.939'",
 "'2025-03-26T20:11:37.957

In [ ]:
# extract the values for the plots
df_columns = df.columns.tolist()
# these are the column names for each frame
value_columns = [f for f in df_columns if f.startswith('Frame-')]
frame_count = len(value_columns)
# these are the frame numbers
value_frames = [f.replace('Frame-', '') for f in value_columns]

# x axis texts -------------------------------------------------------
xtext = '[UTC] Start date and time of the observati'
try:
    df_plot = df[df['Description'] == xtext].reset_index(drop=True)
    xtext = df_plot.iloc[0][value_columns].values.astype(str).tolist()
    xtext = [x[12:20] for x in xtext]
    xtext = [f'{value_frames[i]} : {x}' for i, x in enumerate(xtext)]
except Exception as e:
    print(f"Error filtering DataFrame: {e}")

# y axis values ------------------------------------------------------
flt = [#'Effective mean airmass', 
       #'[mbar] Atmospheric pressure',
       #'[%] Current percentage humidity',
       #'[deg C] External temperature',
       #'[deg] Enclosure azimuth',
       #'km wind speed',
       '[arcsec] Frame FWHM in arcsec',
       #'[arcsec] Autoguider FWHM',
       'Mean image ellipticity (1-B',
       #'[deg] PA of mean image ellipticity',
       ]

try:
    df_plot = df[df['Description'].isin(flt)].reset_index(drop=True)

except Exception as e:
    print(f"Error processing DataFrame: {e}")


In [52]:
fig = go.Figure()
x = list(range(1, 212))
color_list = ['blue', 'green', 'brown', 'red', 'red', 'brown']

try:
    for i in range(len(df_plot)):
        yvalue = df_plot.iloc[i]['Description']
        print(f"Processing: {yvalue}")

        list_vals = df_plot.iloc[i][value_columns].values.astype(float).tolist()
        y, miny, maxy = get_normalised(list_vals, verbose = False)
        yname = f'{yvalue} {miny:.2f} - {maxy:.2f}'

        if yvalue == 'km wind speed':
            fig.add_trace(go.Scatter(x=x, y=y, name=yname,
                                    fill='tozeroy',  # Fill area down to y=0
                                    mode='none',      # No line or markers
                                    fillcolor='rgba(0, 100, 250, 0.4)',
                                    text=xtext,           # This sets hover text per point
                                    hoverinfo='text+y'))  # Semi-transparent blue
        else:
            fig.add_trace(go.Scatter(x=x, y=y, name=yname,
                                    line=dict(color=color_list[i]),
                                    text=xtext,           # This sets hover text per point
                                    hoverinfo='text+y'))     # Show only the custom text and y value))

    # Set the tick labels using xtexts
    fig.update_layout(
        xaxis=dict(
            tickmode='array',
            tickvals=x[::10],       # Every 10th x value
            ticktext=xtext[::10],  # Matching every 10th label
            tickangle=45            # Rotate labels 45 degrees
        ),
        title=f'BANZAI Headers measures for {star}, {obs_date}',
    )
    fig.show()
except Exception as e:
    print(f"Error creating plot: {e}")

Processing: [arcsec] Frame FWHM in arcsec
Processing: Mean image ellipticity (1-B
